# 03 · Models and honest validation: a guided tour of Phase 3

Phase 2 turned hands into one row of features per pair. Phase 3 trains the four models. More importantly, it works out **how much to believe the scores they get**. Most of the lessons here are about validation, not about LightGBM.

Same structure as notebooks 01 and 02: 🎯 Question · 🤔 Why it matters · 🔬 Method · 📊 What we saw · 💡 Inference · ➡️ Consequence.

### Sections
1. What has to be built, and how it is graded
2. Why a 0.99 AP means almost nothing: building honest validation sets
3. The naive investigators: baselines B0 and B1
4. M1 risk model: what it learned, and the hidden colluders it found
5. The weak family: diagnosing coordinated isolation, and the F8 features that fixed it
6. The exposure trap: development vs evaluation, and training on windows
7. M2 behavior model, and why predicting `none` almost never helps
8. M4 evidence ranker: why a second, hand-level model is needed
9. M3 and `other_coordination`
10. What drives the risk score: importance and ablations
11. Summary, what to expect on Kaggle, and self-check

> Outputs use competition data, so they stay local. Run `python -m suspoker.pipeline --stage S5`, `--stage S5W`, then `python -m suspoker.train` (and optionally `python scripts/ablate.py`) first.

In [ ]:
import json

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from suspoker.config import ARTIFACTS_DIR, MODELS_DIR, RAW_DIR
from suspoker.features import CATALOG
from suspoker.metric import average_precision, average_precision_at_5
from suspoker.modeling import FAMILIES, KEYS, PAIR_FEATURES, keyed, load_dev, matrix, with_exposure
from suspoker.pipeline import FEATURES_DIR

pl.Config(tbl_rows=30, tbl_width_chars=180, fmt_str_lengths=60, float_precision=3)
results = json.loads((ARTIFACTS_DIR / "results.json").read_text())
oof = pl.read_parquet(ARTIFACTS_DIR / "oof" / "pairs.parquet")
oof_ev = pl.read_parquet(ARTIFACTS_DIR / "oof" / "evidence.parquet")
pf = pl.read_parquet(FEATURES_DIR / "pair_features.parquet")
windows = pl.read_parquet(FEATURES_DIR / "pair_features_windows.parquet")
d = load_dev(pf, windows)
print(f"mirror pairs: {oof.height:,} | labeled: {oof['labeled'].sum():,} | positives: {oof['label'].sum()}")
print(f"pair features: {results['n_pair_features']} | hand features: {results['n_hand_features']}")

## 1 · What has to be built, and how it is graded

| Output per evaluation pair | Model | Weight in the score |
|---|---|---|
| `risk_score`, ranked over all 112,540 pairs | **M1** LightGBM binary | **70%** (Pair AP) |
| five evidence hands | **M4** LightGBM at hand level | 20% (Evidence MAP@5) |
| `predicted_behavior` | **M2** 3-class LightGBM (+ **M3** for `other_coordination`) | 10% (Behavior MAP) |

🤔 **The catch:** Average Precision depends on the **base rate**. If 20% of the pairs are positive, a mediocre ranking still scores well. If 0.3% are positive, every false positive near the top costs a lot. Our labeled set is 20% positive; the evaluation set is a fraction of a percent. Section 2 shows how big that difference is.

In [ ]:
lab = oof.filter(pl.col("labeled"))
y_l, s_l = lab["label"].to_numpy(), lab["risk_raw"].to_numpy()
y_m, s_m = oof["label"].to_numpy(), oof["risk_raw"].to_numpy()
print("same model, same out-of-fold scores:")
print(f"  AP on the labeled pairs (20% positive): {average_precision(y_l, s_l):.3f}")
print(f"  AP on the mirror set   (0.3% positive): {average_precision(y_m, s_m):.3f}")
print(f"  a random ranking would get about {y_l.mean():.3f} and {y_m.mean():.4f}")

📊 **What we saw:** the same scores give AP ≈ 0.99 on the labeled pairs and ≈ 0.70 on the realistic set.

💡 **Inference:** almost nothing in the labeled set looks like a colluder by accident, because it holds only 1,488 negatives. Among 120,000 ordinary pairs, a few hundred will look suspicious by chance and get ranked above some real colluders.

➡️ **Consequence:** never tune on labeled-set AP alone. We need a validation set with a realistic base rate.

## 2 · Building honest validation sets

### 2a · Folds grouped by table
🎯 Each table's 30 players play only each other. A model that sees some pairs from a table during training learns *those players' habits*, and its score on the other pairs from that table is inflated. The evaluation pairs come from the same tables as the development labels, but the organizer **removed every pair that contains a labeled colluder**, so for colluders it is effectively unseen players.

🔬 `StratifiedGroupKFold(5)`: group = `table_id` (no table on both sides) and stratify = behavior family (the leaderboard split is stratified by family too).

In [ ]:
f = d.labeled.group_by("fold").agg(pl.len().alias("pairs"), pl.col("label").sum().alias("positives"),
                                   pl.col("table_id").n_unique().alias("tables"),
                                   *[(pl.col("behavior_family") == fam).sum().alias(fam[:12]) for fam in FAMILIES])
print(f.sort("fold"))
print("tables appearing in more than one fold:",
      d.mirror.group_by("table_id").agg(pl.col("fold").n_unique()).filter(pl.col("fold") > 1).height)

### 2b · Rebuilding the organizer's evaluation filter
🎯 Can we build a development-period set that looks like the evaluation set?

🔬 `rawkaggle.md` says publicly labeled pairs and pairs containing a labeled positive player are excluded. We apply that to the evaluation-period pairs and check the minimum shared hands (38).

In [ ]:
labels = keyed(pl.read_csv(RAW_DIR / "development_labels.csv"))
evals = keyed(pl.read_csv(RAW_DIR / "evaluation_pairs.csv"))
pos = labels.filter(pl.col("label") == 1)
pos_players = pl.concat([pos["player_1"], pos["player_2"]]).unique().implode()
cand = (pf.filter(pl.col("phase") == 1).join(labels.select(KEYS), on=KEYS, how="anti")
        .filter(~pl.col("player_lo").is_in(pos_players) & ~pl.col("player_hi").is_in(pos_players)))
print(f"evaluation-period pairs after the exclusions, with >= 38 shared hands: "
      f"{cand.filter(pl.col('n_shared') >= 38).height:,}   (official evaluation_pairs.csv: {evals.height:,})")
print("smallest shared-hand count among labeled development pairs:",
      labels.join(pf.filter(pl.col("phase") == 0), on=KEYS)["n_shared"].min(), "= 38 x 3,000/2,000")

📊 **What we saw:** 112,569 vs 112,540. The filter is reproduced almost exactly. Labeled development pairs all have ≥ 57 shared hands, which is the same 38-hand rule scaled to the 3,000-hand development period.

### 2c · Are the confirmed negatives special?
If the organizer had hand-picked *hard* negatives, the labeled set would be a strange sample. We checked by training a classifier to tell confirmed negatives from random unlabeled pairs: **AUC 0.62**, and most of that came from their ≥ 57 shared-hand cutoff (DEC-018). So they are ordinary pairs.

💡 **Inference:** a fair stand-in for the evaluation set is "labeled pairs + all unlabeled development pairs passing the same filter", with unlabeled pairs counted as negatives.

➡️ **Consequence: three validation sets** (DEC-018):

| Set | Pairs | Positive | Features | Role |
|---|---|---|---|---|
| labeled | 1,860 | 20% | full development period | sanity check |
| mirror | 122,580 | 0.30% | full development period | realistic base rate |
| **window** | 122,580 | 0.30% | development hands 1000–2999 | **realistic base rate *and* exposure → main leaderboard proxy** (Section 6) |

Mirror and window are **pessimistic**, because some unlabeled pairs are real colluders whose labels are hidden. Section 4 shows they exist.

## 3 · The naive investigators: B0 and B1
🎯 What does a simple rule achieve? **B0** ranks pairs by how many hands they shared. **B1** ranks by the absolute net chip flow between them (bb per 100 hands) and uses the biggest flows as evidence. MA3 requires every model component to beat B1.

In [ ]:
rows = []
for name, r in results["models"].items():
    for vset in ("labeled", "mirror", "window"):
        s = r[vset]
        rows.append({"model": name, "set": vset, "final": s["final"], "pair_ap": s["pair_ap"],
                     "evidence_map": s["evidence_map"], "behavior_map": s["behavior_map"]})
res = pl.DataFrame(rows)
print(res.filter(pl.col("set") == "window").drop("set"))

📊 **What we saw:** B0 is at chance: sitting together often is not suspicious, and "repeated opponent selection" is one of the organizer's decoys. B1 has some signal on the labeled set but collapses at the realistic base rate, because big chip flows also come from ordinary big pots and tilt. The full model beats B1 on all three components by a wide margin.

💡 **Inference:** the chip-flow story ("one player is losing to the other") is not enough. The decoys are built to look exactly like that.

## 4 · M1, the risk model
🔬 LightGBM binary on the pair features: 15 leaves, 500 trees at learning rate 0.02, averaged over 3 seeds. It trains on labeled pairs **plus 5,000 unlabeled development pairs per view as noisy negatives** (DEC-019). The unlabeled pairs show the model what ordinary pairs look like; at a < 1% hidden-positive rate, the label noise is small. That raised mirror AP from 0.59 to 0.62.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
groups = {"labeled positives": oof.filter(pl.col("label") == 1),
          "confirmed negatives": oof.filter(pl.col("labeled") & (pl.col("label") == 0)),
          "unlabeled": oof.filter(~pl.col("labeled"))}
bins = np.linspace(0, 1, 41)
for name, g in groups.items():
    ax.hist(g["risk_raw"].to_numpy(), bins=bins, density=True, histtype="step", lw=2, label=f"{name} ({g.height:,})")
ax.set_yscale("log"); ax.set_xlabel("OOF risk (raw)"); ax.set_ylabel("density (log)"); ax.legend()
plt.tight_layout(); plt.show()
neg = groups["confirmed negatives"]["risk_raw"]; unl = groups["unlabeled"]["risk_raw"]
for th in (0.99, 0.9, 0.5):
    exp = (neg > th).mean() * unl.len()
    print(f"risk > {th}: unlabeled {int((unl > th).sum()):>4}  vs  {exp:6.1f} expected if they behaved like confirmed negatives")

📊 **What we saw:** positives pile up near 1 and negatives near 0. Some unlabeled pairs score above 0.99, a level that **no** confirmed negative reaches, and there are more high-scoring unlabeled pairs than the negatives' rate predicts.

💡 **Inference:** these are almost certainly **hidden colluders**: the organizer said the labels do not list every coordinated pair. That is why the mirror and window scores are pessimistic: each hidden colluder near the top is counted as a false positive.

➡️ **Consequence:** the true leaderboard AP could be higher than the window estimate. The Kaggle submission (P4) will tell us.

## 5 · The weak family: coordinated isolation
🎯 Which kind of collusion does M1 miss?

In [ ]:
def family_ap(df, score_col):
    negs = df.filter(pl.col("label") == 0)
    out = {}
    for fam in FAMILIES:
        p = df.filter(pl.col("behavior_family") == fam)
        y = np.r_[np.ones(p.height), np.zeros(negs.height)]
        out[fam] = average_precision(y, np.r_[p[score_col].to_numpy(), negs[score_col].to_numpy()])
    return out
print("window-set AP of each family against all negatives:",
      {k: round(v, 3) for k, v in family_ap(oof, "risk_window_raw").items()})

📊 **Before F8** (first M1), the per-family mirror APs were directed transfer 0.53, soft play 0.49, and **coordinated isolation 0.13**. A third of isolation pairs scored below 0.2.

🔬 **Diagnosis by reading hands.** We printed several planted isolation evidence hands (locally; they are never published). They all followed one pattern: **one partner open-raises a weak hand** (K4 offsuit, T3, K2) **and the other partner folds behind**, often while a third player is in the pot. Our F3–F5 features look at how a player *responds to* the partner's bets. Here the partner never bets; their mere **presence** changes how the other one opens.

🔬 **F8 presence features** (DEC-020): for each player, the shrunk lift of VPIP, PFR, open-raise, weak open (bottom half of hands) and weak VPIP in hands **shared with the partner** vs hands **without** them. Same lift principle, different condition.

In [ ]:
dl = d.labeled
pres = [c for c in PAIR_FEATURES if "_pres_" in c]
rows = []
for c in pres:
    x = dl[c].fill_null(0).to_numpy()
    row = {"feature": c}
    for fam in FAMILIES:
        yf = (dl["behavior_family"] == fam).to_numpy().astype(int)
        row[fam] = max(average_precision(yf, x), average_precision(yf, -x))
    rows.append(row)
print("single-feature AP per family on the labeled pairs (in-sample; family base rates ~0.05-0.08):")
print(pl.DataFrame(rows).sort("coordinated_isolation", descending=True))

📊 **What we saw:** `pfr_pres_min` and `wopen_pres_min` reach single-feature AP ≈ 0.7–0.8 for isolation, more than ten times the isolation base rate (~0.05). Adding F8 raised isolation mirror AP from 0.13 to 0.30 and overall mirror AP from 0.62 to 0.70.

💡 **Inference:** *min* is what works. **Both** partners raise more when the other is seated, because each one plays the opener role in turn. A single loose player only moves *max*.

➡️ **Lesson:** when a model is weak on one class, read examples of that class. The fix was a missing *kind* of feature, not tuning.

## 6 · The exposure trap
🎯 The model trains on 3,000-hand development features but is scored on 2,000-hand evaluation features. Does that matter?

🔬 **Adversarial validation:** train a classifier to tell evaluation pairs from development pairs. If it can, the two look different to a model.

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_predict

ev_pf = evals.join(with_exposure(pf.filter(pl.col("phase") == 1)), on=KEYS).sample(15000, seed=0)
views = {"full development period": d.mirror.filter(~pl.col("labeled")),
         "2,000-hand window": d.windows[1000].filter(~pl.col("labeled")).drop_nulls("n_shared_frac")}
for name, dv in views.items():
    dv = dv.sample(15000, seed=0)
    X = np.vstack([matrix(ev_pf, PAIR_FEATURES), matrix(dv, PAIR_FEATURES)])
    y = np.r_[np.ones(15000), np.zeros(15000)]
    p = cross_val_predict(lgb.LGBMClassifier(n_estimators=150, num_leaves=15, verbose=-1), X, y, cv=5,
                          method="predict_proba")[:, 1]
    print(f"evaluation pairs vs {name:<24} AUC = {roc_auc_score(y, p):.3f}")

📊 **What we saw:** against full-period development features the classifier reaches **AUC ≈ 0.95**: the two sets are almost perfectly separable. Against features recomputed on a 2,000-hand window it drops to **≈ 0.67**.

💡 **Inference:** the difference is mostly *exposure*. With fewer hands, shrinkage pulls lifts harder toward 0, block maxima come from 4 blocks instead of 6, and rates are noisier. A model trained on 3,000-hand features expects bigger lifts than evaluation pairs will show.

🔬 **The experiment** (DEC-020): train on one exposure, test on another (OOF Pair AP, mirror pairs).

| Train on | Test on full period | Test on 2,000-hand window |
|---|---|---|
| full period only | 0.695 | **0.621** (−0.07) |
| windows only | — | 0.647 |
| full + both windows | 0.697 | 0.645 |

➡️ **Consequence:** new stage **S5W** recomputes development features on hands 0–1999 and 1000–2999. M1 and M2 train on all three views at once (the same pair's views always share a fold, so nothing leaks). Raw counts were replaced by shares of the period (`n_shared_frac`). The **window set** is the main leaderboard proxy.

## 7 · M2 behavior, and the `none` question
🔬 M2: a 3-class LightGBM on the 372 positives (× 3 views). Each pair gets `argmax` of its family probabilities.

In [ ]:
p = oof.filter(pl.col("label") == 1)
pred = p.select(pl.concat_list([f"p_{f}_window" for f in FAMILIES]).list.arg_max().alias("k"))["k"].to_numpy()
true = p["behavior_family"].replace_strict({f: i for i, f in enumerate(FAMILIES)}).to_numpy()
cm = np.zeros((3, 3), int)
np.add.at(cm, (true, pred), 1)
print("rows = true family, columns = predicted (OOF, window view)")
print(pl.DataFrame(cm, schema=[f[:14] for f in FAMILIES]).with_columns(pl.Series("true", [f[:14] for f in FAMILIES])))
print(f"family accuracy among positives: {np.trace(cm) / cm.sum():.3f}")
print("none-threshold tuning (window Behavior MAP):",
      [(round(t['none_below'], 4), round(t['window_behavior_map'], 4)) for t in results["behavior_none_tuning"]])

🎯 **Should low-risk pairs be labeled `none`?** In Behavior MAP, family X's ranking is "every pair predicted X, ordered by risk", followed by a **tie pile at score 0** ordered by pair_id, which is effectively random.

💡 **Reasoning** (DEC-021): moving pairs below a risk threshold to `none` only changes the order **below** the threshold. It swaps a tail ordered by risk for random order in the pile. That helps only if the risk ordering of the tail is *worse than random*, which it isn't. The tuning table shows this: the best threshold gains essentially nothing.

➡️ **Consequence:** argmax for every pair; a threshold is adopted only if it gains > 0.005. (An earlier audit note said `none` "removes false positives", which is true only for false positives that sit below every positive they would protect.)

⚠️ **A validation bug worth remembering.** In the first version, unlabeled pairs got ids starting with `~`, which sorts after every real `P…` id. In every tie, labeled pairs came first, which inflated the tie pile. For example B0, which never predicts soft play, got soft-play AP 0.097. Synthetic ids are now hash-based in the official format, so they interleave randomly. **When a metric breaks ties by id, your ids are part of the experiment.**

## 8 · M4, the evidence ranker
🎯 Why not just pick the pair's biggest pots? Evidence MAP@5 asks for the **specific hands** where the planted behavior happened, out of ~57 candidate hands per pair. The pair-level model knows *that* a pair colludes, not *where*.

🔬 M4 is LightGBM at **hand level**, trained only on the candidate hands of positive pairs (evidence is only scored for true colluders, so only the ranking *within* a pair matters). Features are per-hand versions of F2–F5/F8, within-pair context (rank of the pot among the pair's hands, the pair's flag rate in the same time block), and M2's OOF family probabilities, since what counts as evidence depends on the behavior.

In [ ]:
ev = oof_ev.sort(["pair_id", "score"], descending=[False, True])
fam = d.labeled.select("pair_id", "behavior_family")
rng = np.random.default_rng(0)
rows = []
for (pid,), g in ev.group_by(["pair_id"], maintain_order=True):
    rel = set(g.filter(pl.col("y") == 1)["hand_id"])
    hands = g["hand_id"].to_list()
    rand = [hands[i] for i in rng.permutation(len(hands))[:5]]  # a fresh random order for every pair
    rows.append({"pair_id": pid, "m4": average_precision_at_5(rel, hands[:5]),
                 "random": average_precision_at_5(rel, rand)})
e = pl.DataFrame(rows).join(fam, on="pair_id")
print(e.group_by("behavior_family").agg(pl.len().alias("pairs"), pl.col("m4").mean(), pl.col("random").mean()))
print(f"overall: M4 {e['m4'].mean():.3f} | random {e['random'].mean():.3f} | candidates per pair {ev.height / e.height:.0f}")

📊 **What we saw:** M4 gets MAP@5 ≈ 0.41, against ≈ 0.18 for B1's "biggest flows" and far less for random order (table above). Soft play and directed transfer are found well (the planted action is distinctive: a strong hand checked down, a fold while ahead). Isolation is hardest (≈ 0.20), because "open, partner folds" also happens constantly in honest play.

💡 **Inference:** a pair might have more manipulated hands than the 5 listed. Any unlisted ones count as "wrong" here, so this number is a lower bound on how well M4 finds manipulated hands.

## 9 · M3 and `other_coordination`
The organizer hides a fourth mechanism that never appears in the labels. M3 is an Isolation Forest on the lift and presence features. A pair is labeled `other_coordination` only if it is in the top 1% for novelty, M2 is unsure of its family (max probability < 0.5), and its risk is above the median.

In [ ]:
print(results["other_coordination"])

📊 A few dozen pairs get the label and none of them is a labeled positive. Since `other_coordination` is excluded from Behavior MAP, the label costs nothing measurable and might catch the hidden mechanism. It stays first in the scope-cut order.

## 10 · What drives the risk score?

In [ ]:
boosters = sorted(MODELS_DIR.glob("m1_risk_s*.txt"))
gain = np.mean([lgb.Booster(model_file=str(b)).feature_importance("gain") for b in boosters], axis=0)
names = lgb.Booster(model_file=str(boosters[0])).feature_name()
def fam_of(n):
    return CATALOG[n][0] if n in CATALOG else ("F1 exposure" if n == "n_shared_frac" else "F7 controls")
imp = pl.DataFrame({"feature": names, "gain": gain}).with_columns(
    (pl.col("gain") / pl.col("gain").sum()).alias("share"),
    pl.col("feature").map_elements(fam_of, return_dtype=pl.String).alias("family"))
print(imp.sort("gain", descending=True).head(15).select("feature", "family", "share"))
print(imp.group_by("family").agg(pl.col("share").sum()).sort("share", descending=True))

In [ ]:
abl = ARTIFACTS_DIR / "ablation.csv"
if abl.exists():
    print("drop one feature family, retrain M1 out of fold (Pair AP):")
    print(pl.read_csv(abl).select("dropped", "n_features", "window_ap", "window_delta", "mirror_ap", "mirror_delta"))
else:
    print("run `python scripts/ablate.py` to produce the ablation table")

💡 **How to read the two views.** *Gain* says what the model **uses**. *Ablation* says what it **needs**: dropping a family whose information is duplicated elsewhere costs little even if its gain is high. A family with a big ablation loss carries unique signal.

## 11 · Summary

| # | Finding | Why it matters |
|---|---|---|
| 1 | Labeled-set AP ≈ 0.99 but realistic AP ≈ 0.66–0.70 for the same model | AP depends on the base rate, so validate at the evaluation's base rate |
| 2 | The evaluation filter can be reproduced (112,569 vs 112,540) | Enables a faithful local stand-in for the leaderboard |
| 3 | Unlabeled pairs as noisy negatives: +0.03 AP | Ordinary pairs teach the model what "normal" looks like |
| 4 | Reading isolation hands → F8 presence features: isolation AP 0.13 → 0.30 | Weak class → look at examples, find the missing kind of feature |
| 5 | Exposure shift (AUC 0.95) costs 0.07 AP; training on 2,000-hand windows recovers part of it | Train at the exposure you will be tested on |
| 6 | `none` thresholds can't beat a risk-ordered tail | Think through the metric before tuning |
| 7 | Synthetic ids changed tie order and inflated scores | Ids are part of the experiment when ties are broken by id |
| 8 | M4 evidence MAP@5 ≈ 0.41 vs 0.05 random | A hand-level model is needed to say *where* the collusion is |

**Expectation for Kaggle (P4):** window-set final ≈ 0.61. Hidden colluders make that pessimistic. The remaining dev→eval differences (style drift, the evaluation's own ≥ 38 filter) make it optimistic. A leaderboard score anywhere from ~0.55 to ~0.70 would be consistent with this analysis.

### Self-check
1. Why does the same model get AP 0.99 on one set and 0.70 on another, with no change in its scores?
2. Why must all three exposure views of a pair be in the same fold?
3. Unlabeled pairs are "unknown, not negative". Why is it still acceptable to train on them as negatives, and why must we *not* report them as true negatives?
4. Why is `pfr_pres_min` better than `pfr_pres_max` for isolation?
5. In Behavior MAP, why can't a low-risk `none` threshold help much?

<details><summary>Answers</summary>

1. AP measures precision along the ranking. With 1,488 negatives, few of them can outrank the colluders. With 120,000, many ordinary pairs look suspicious by chance and get in front of some colluders. The scores are the same; the competition among negatives is not.
2. The views are the same players at different time spans. If one view trains and another tests, the model is tested on players it has already seen, which inflates the score.
3. Hidden positives are rare (< 1%), so as training labels they add only a little noise, and they teach the model what ordinary pairs look like. As *evaluation* labels they would make us claim errors that might be correct detections, so mirror and window scores are explicitly called pessimistic.
4. In isolation *both* partners open more lightly when the other is seated. A single loose or tilting player raises only the max. The min is high only when both change, which is the coordination signature.
5. The pairs it moves to `none` are all below the threshold, so it can only reorder the bottom of the list, swapping a risk-ordered tail for the random tie order at score 0. Random is no better than a ranking that has real signal.
</details>